# 12｜深層神經網路訓練_梯度與遷移

用梯度範數與深度尺度理解消失、爆炸與裁剪。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/12_深層神經網路訓練_梯度與遷移.md`，目前 50 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

激活/初始化、BN/LN、凍結/解凍、線性預訓練與圖片描述介面。

各格會標示實驗資料、評估方式與適用範圍；架構圖用於說明機制與張量形狀。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 12-03 | 12、14、37、38 | 課堂小實驗 |
| 12-04 | 3、14 | 執行結果圖 |
| 12-05 | 13、15、16、17、18、20、21、22、23、24、25、26、27、50 | 1. 激活函數、初始化與門控 |
| 12-06 | 28、29、30、31、32、33、34、35、50 | 2. BatchNorm 與 LayerNorm 的統計軸 |
| 12-07 | 5、6、7、8、9、10、11、37、38、39、40、41、42、43、44、45、46、47、48、49、50 | 3. 可執行的凍結、解凍與預訓練流程 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 12-01｜投影片 12：2 頁
import importlib.util, subprocess, sys
_needed = {'scipy': 'scipy>=1.11,<2', 'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 12-02｜投影片 12：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 課堂小實驗

$(1/4)^L$ 只界定 sigmoid 導數連乘，未包含權重矩陣，因此不是整個網路梯度的上界。

In [ ]:
# 程式 12-03｜投影片 12：12、14、37、38 頁
depth = np.array([1, 2, 5, 10, 20])
sigmoid_upper = 0.25 ** depth
scale_table = pd.DataFrame({'depth': depth, 'sigmoid_derivative_upper_bound': sigmoid_upper})

gradient = np.array([3.0, 4.0, 12.0])
before = np.linalg.norm(gradient)
clip_norm = 5.0
clipped = gradient * min(1.0, clip_norm / before)
after = np.linalg.norm(clipped)
print(f'gradient norm: {before:.1f} -> {after:.1f}')
scale_table

## 執行結果圖

In [ ]:
# 程式 12-04｜投影片 12：3、14 頁
plt.semilogy(depth, sigmoid_upper, 'o-', label=r'$(1/4)^L$')
plt.axhline(1e-6, color='tab:red', ls='--', label='1e-6')
plt.xlabel('number of sigmoid derivative factors')
plt.ylabel('upper bound of derivative product')
plt.title('Repeated small derivatives shrink the backward signal')
plt.legend()
savefig('12_gradient_transfer_demo')
report('deep_gradient_transfer', {'gradient_norm_before': before, 'gradient_norm_after': after,
                                  'depths': depth.tolist(), 'sigmoid_upper': sigmoid_upper.tolist()})

## 1. 激活函數、初始化與門控

NumPy 畫出所有主要曲線；導數用有限差分讀圖（折點不作光滑導數主張）。初始化是標準差與 fan-in／fan-out 的配對，不是全層填同一抽樣值。

In [ ]:
# 程式 12-05｜投影片 12：13、15、16、17、18、20、21、22、23、24、25、26、27、50 頁
from scipy.special import expit,ndtr
z12=np.linspace(-5,5,501);elu12=np.where(z12>=0,z12,np.expm1(z12))
acts12={'sigmoid':expit(z12),'ReLU':np.maximum(z12,0),'Leaky(.2)':np.where(z12>=0,z12,.2*z12),
 'ELU':elu12,'SELU':1.050700987*np.where(z12>=0,z12,1.673263242*np.expm1(z12)),
 'GELU':z12*ndtr(z12),'SiLU':z12*expit(z12),'Mish':z12*np.tanh(np.logaddexp(0,z12)),'ReLU2':np.maximum(z12,0)**2}
fig,axes=plt.subplots(2,2,figsize=(11,7))
for ax,names in zip(axes.flat,[['sigmoid'],['ReLU','Leaky(.2)'],['ELU','SELU'],['GELU','SiLU','Mish','ReLU2']]):
    for name in names:ax.plot(z12,acts12[name],label=name)
    ax.legend();ax.set(xlabel='z',ylabel='activation')
savefig('c12_activation_family')
print('Glorot / He / LeCun std:',np.sqrt([2/150,2/100,1/100]))
rng12=np.random.default_rng(SEED);input12=rng12.normal(size=(300,100))
fig,ax=plt.subplots()
for name,scale in [('small',.01),('large',.3),('He',np.sqrt(2/100))]:
    h12=input12.copy();moments12=[]
    for layer12 in range(15):h12=np.maximum(h12@rng12.normal(0,scale,(100,100)),0);moments12.append(np.mean(h12**2))
    ax.semilogy(range(1,16),moments12,label=name)
ax.set(xlabel='layer',ylabel='activation second moment');ax.legend();savefig('c12_initialization_signal')
# SwiGLU: 兩個獨立投影，再逐元素相乘
x12=rng12.normal(size=(4,8));gate12=x12@rng12.normal(size=(8,12));value12=x12@rng12.normal(size=(8,12))
swiglu12=(gate12*expit(gate12))*value12
assert swiglu12.shape==(4,12)
print('SwiGLU input / hidden / projected output:',x12.shape,swiglu12.shape,(swiglu12@rng12.normal(size=(12,8))).shape)

## 2. BatchNorm 與 LayerNorm 的統計軸

NumPy 的變異用 ddof=0 對應前向正規化；PyTorch running_var 的更新另有無偏估計細節，此處不冒充逐位元相同。BN 推論使用保存統計，LN 每筆自行計算。

In [ ]:
# 程式 12-06｜投影片 12：28、29、30、31、32、33、34、35、50 頁
a12=np.array([[1.,2.,3.,4.],[2.,4.,6.,8.],[3.,6.,9.,12.],[4.,8.,12.,16.]])
eps12=1e-5
bn12=(a12-a12.mean(0,keepdims=True))/np.sqrt(a12.var(0,keepdims=True)+eps12)
ln12=(a12-a12.mean(1,keepdims=True))/np.sqrt(a12.var(1,keepdims=True)+eps12)
assert np.allclose(bn12.mean(0),0) and np.allclose(ln12.mean(1),0)
one12=np.array([1.,2.,3.,4.]);norm12=(one12-one12.mean())/np.sqrt(one12.var()+eps12)
print('Hand example mean / variance / output:',one12.mean(),one12.var(),norm12)
print('BN first feature / LN first sample:',bn12[:,0],ln12[0])
running_mean12=np.zeros(4);running_var12=np.ones(4)
running_mean12=.9*running_mean12+.1*a12.mean(0);running_var12=.9*running_var12+.1*a12.var(0)
inference12=(a12-running_mean12)/np.sqrt(running_var12+eps12)
image12=rng12.normal(size=(2,3,4,4));stats12=image12.mean(axis=(0,2,3),keepdims=True)
assert stats12.shape==(1,3,1,1)
fig,axes=plt.subplots(1,3,figsize=(10,3))
for ax,mat,title in zip(axes,[a12,bn12,ln12],['input','BN: columns','LN: rows']):ax.imshow(mat,aspect='auto',cmap='coolwarm');ax.set_title(title)
savefig('c12_normalization_axes')

## 3. 可執行的凍結、解凍與預訓練流程

合成二維分類先訓練來源任務，再複製骨幹換新頭。比較參數快照才能證明凍結。這不是 Fashion MNIST 成效，也沒有宣稱已訓練圖片描述模型；圖片 encoder 到 token decoder 只驗算介面。

In [ ]:
# 程式 12-07｜投影片 12：5、6、7、8、9、10、11、37、38、39、40、41、42、43、44、45、46、47、48、49、50 頁
from scipy.special import logsumexp
import copy
x12=rng12.normal(size=(240,2));ya12=(x12[:,0]>0).astype(int);yb12=(x12[:,0]+.4*x12[:,1]>0).astype(int)
params12={'W':rng12.normal(0,.5,(2,8)),'b':np.zeros(8),'V':rng12.normal(0,.2,(8,2)),'c':np.zeros(2)}
def train12(par,x,y,epochs,lr_body,lr_head):
    losses=[]
    for _ in range(epochs):
        z=x@par['W']+par['b'];h=np.maximum(z,0);logits=h@par['V']+par['c']
        losses.append(np.mean(logsumexp(logits,axis=1)-logits[np.arange(len(y)),y]))
        p=np.exp(logits-logsumexp(logits,axis=1,keepdims=True));p[np.arange(len(y)),y]-=1;p/=len(y)
        dh=(p@par['V'].T)*(z>0)
        grads={'W':x.T@dh,'b':dh.sum(0),'V':h.T@p,'c':p.sum(0)}
        # backward -> clip -> step；每步重算梯度，沒有跨步累積
        norm=np.sqrt(sum(np.sum(g*g) for g in grads.values()));scale=min(1,5/max(norm,1e-12))
        for k,g in grads.items():par[k]-=(lr_body if k in ['W','b'] else lr_head)*scale*g
    return losses
source12=train12(params12,x12[:160],ya12[:160],100,.05,.05)
target12=copy.deepcopy(params12);target12['V']=rng12.normal(0,.2,(8,2));snapshot12=target12['W'].copy()
head12=train12(target12,x12[:40],yb12[:40],60,0,.1);assert np.array_equal(snapshot12,target12['W'])
fine12=train12(target12,x12[:40],yb12[:40],60,.01,.05);assert not np.array_equal(snapshot12,target12['W'])
hold_logits12=np.maximum(x12[160:]@target12['W']+target12['b'],0)@target12['V']+target12['c']
print('Target held-out accuracy (synthetic):',np.mean(hold_logits12.argmax(1)==yb12[160:]))
plt.plot(source12,label='source train');plt.plot(head12+fine12,label='target head then unfreeze');plt.axvline(60,ls='--',color='gray');plt.legend();plt.xlabel('update');plt.ylabel('training CE');savefig('c12_transfer_training')
# 無監督線性 autoencoder 預訓練：只用訓練 X 的 SVD 學表示
center12=x12[:160].mean(0);_,_,vt12=np.linalg.svd(x12[:160]-center12,full_matrices=False)
code12=(x12[:160]-center12)@vt12[:1].T;recon12=code12@vt12[:1]+center12
print('Unsupervised representation / reconstruction MSE:',code12.shape,np.mean((x12[:160]-recon12)**2))
# 圖片 encoder 輸出接簡化 recurrent decoder：形狀示範，無語意訓練
encoded12=rng12.normal(size=(2,8));state12=encoded12.copy();tokens12=[]
for t12 in range(4):
    state12=np.tanh(state12@np.eye(8)*.7);tokens12.append((state12@rng12.normal(size=(8,6))).argmax(1))
print('Image feature -> token IDs shape:',encoded12.shape,np.stack(tokens12,axis=1).shape)

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。